# NeuroVision-X — Kaggle training driver

Thin driver. No training logic lives here: this notebook attaches the code and the data,
composes the real Hydra config, and calls `scripts/train.py::run_training`. Everything else
is in `src/neurovision/`, tested on the Mac's CPU.

**Before running:** notebook settings → GPU accelerator ON, internet ON (both need a
phone-verified account). Attach the preprocessed dataset. For a long run use
*Save Version → Save & Run All (Commit)*, never the interactive session.

Every cell below cell 1 fails immediately and with a readable message if a path is wrong —
the point is to find out in the first minute, not 40 minutes in.

Full workflow, including how to upload the dataset and chain sessions: `docs/kaggle_workflow.md`.

## 1. Session config — the only cell you edit

In [ ]:
# Note the two different accounts: GitHub is AmishhYadav, Kaggle is amishyadav123.
REPO_URL   = "https://github.com/AmishhYadav/NeuroVision-X.git"
GIT_REF    = "main"
DATA_SLUG  = "amishyadav123/neurovision-brats-prep"  # attached preprocessed dataset
CKPT_SLUG  = None  # previous notebook output / checkpoint dataset to resume from; None = fresh run
EXPERIMENT = "baseline_unet3d"
OVERRIDES  = ["model=unet3d", "training.batch_size=1", "data.dataset_type=dataset", "data.num_workers=2"]

## 2. Code + dependencies

Clone rather than `pip install git+...` alone: `configs/` and `scripts/` are not package data,
and Hydra needs the config tree on disk. The clone is then installed editable with `--no-deps`,
so `import neurovision` works without `PYTHONPATH` — same as local dev.

`torch`/`torchvision` are stripped from `requirements.txt` on purpose: the Kaggle image ships a
CUDA-matched build, and installing the pinned wheel over it silently loses the GPU. The assert
catches that, and a GPU that was never enabled, in ~30 seconds.

If `pip install -e` ever fails on `requires-python` (Kaggle moving off 3.11), replace that line
with `import sys; sys.path.insert(0, "/kaggle/working/repo/src")`.

In [ ]:
!git clone -q --depth 1 -b {GIT_REF} {REPO_URL} /kaggle/working/repo
!grep -vE '^(torch|torchvision)==' /kaggle/working/repo/requirements.txt > /kaggle/working/req-kaggle.txt
!pip install -q -r /kaggle/working/req-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/repo
import torch
assert torch.cuda.is_available(), "No CUDA: GPU accelerator off, or pip replaced Kaggle's CUDA torch build."
print(torch.__version__, torch.cuda.get_device_name(0))

## 3. Environment — W&B key from Kaggle Secrets

Add-ons → Secrets → add `WANDB_API_KEY`, then attach it to this notebook. Never paste the key
into a cell: committed notebook versions are stored with their source.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 4. Resolve the attached dataset

Layout is what `scripts/package_for_kaggle.py` builds: `preprocessed/<case>/{image,label}.npy`,
`metadata.csv`, `splits.yaml`. A wrong or unattached dataset raises here, listing what *is*
mounted so the fix is obvious.

In [ ]:
import shutil
from pathlib import Path

DATA = Path("/kaggle/input") / DATA_SLUG.split("/")[-1]
PREP, SPLITS = DATA / "preprocessed", DATA / "splits.yaml"
missing = [str(p) for p in (DATA, PREP, SPLITS) if not p.exists()]
if missing:
    mounted = sorted(p.name for p in Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(f"Missing {missing}. Add Data -> {DATA_SLUG}. Mounted now: {mounted}")
n_cases = sum(1 for p in PREP.iterdir() if p.is_dir())
if n_cases == 0:
    raise FileNotFoundError(f"{PREP} exists but holds no case directories.")
print(n_cases, "preprocessed cases at", PREP)

## 5. Resume

`/kaggle/input` is read-only and `save_checkpoint` must write, so the previous session's
`last.pt` is copied into `/kaggle/working/checkpoints` first. `select_resume_checkpoint` then
finds it there on its own — the training call is identical for a fresh run and a resume.

With `CKPT_SLUG` set, anything other than exactly one `last.pt` raises. Missing it would not
error during training, it would just silently restart from epoch 0 — the expensive failure this
cell exists to prevent.

In [ ]:
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if CKPT_SLUG:
    hits = sorted((Path("/kaggle/input") / CKPT_SLUG.split("/")[-1]).glob("**/last.pt"))
    if len(hits) != 1:
        raise FileNotFoundError(f"CKPT_SLUG={CKPT_SLUG!r}: want exactly one last.pt, found {[str(h) for h in hits]}. Attach the source, fix the slug, or set CKPT_SLUG=None to start fresh.")
    shutil.copy2(hits[0], CKPT_DIR / "last.pt")
    print("resuming from", hits[0])

## 6. Compose config and train

`hydra.compose` with CLI-style overrides — the same strings `python scripts/train.py a=b` would
take. Calling `run_training` in-process (instead of shelling out) keeps the traceback in the
notebook and lets the cells above hand it already-validated paths.

The log's first line says `FRESH:` or `RESUME: ... from epoch N`. Check it. `max_hours: 11.0`
stops the run cleanly before Kaggle's 12-hour kill.

In [ ]:
import sys

import hydra

sys.path.insert(0, "/kaggle/working/repo/scripts")
from train import run_training

overrides = [f"data.root_dir={DATA}", f"data.preprocessing.out_dir={PREP}", f"data.splits.path={SPLITS}",
             f"training.checkpoint.dir={CKPT_DIR}", f"experiment_name={EXPERIMENT}", *OVERRIDES]
with hydra.initialize_config_dir(version_base="1.3", config_dir="/kaggle/working/repo/configs"):
    cfg = hydra.compose("config", overrides=overrides)
metrics = run_training(cfg)

## 7. Verify the session output

Training already writes into `/kaggle/working/checkpoints`, which is the only path that survives
into the committed version's output — so there is nothing to copy, only to verify. Duplicating a
754 MB SwinUNETR checkpoint elsewhere under `/kaggle/working` would just eat the ~20 GB quota.

Attach this notebook version's output as input to the next session and set `CKPT_SLUG` to it.

In [ ]:
for name in ("last.pt", "best.pt"):
    p = CKPT_DIR / name
    if not p.is_file():
        raise FileNotFoundError(f"{p} missing — nothing to carry into the next session.")
    print(name, f"{p.stat().st_size / 2**20:.0f} MB  epoch={torch.load(p, weights_only=True)['epoch']}")
print(metrics)